# MBTI Personality Type Classification

## Project Overview

This project implements a machine learning pipeline to classify Myers-Briggs Type Indicator (MBTI) personality types based on text posts from users.

### Objectives
- Download and preprocess the MBTI dataset
- Perform text cleaning and feature extraction using TF-IDF
- Train and evaluate three classic ML models: Logistic Regression, SVM, and Naive Bayes
- Handle class imbalance in the dataset
- Analyze model performance using various metrics

### MBTI Types
The MBTI consists of 16 personality types based on four dimensions:
- **E/I**: Extraversion vs Introversion
- **S/N**: Sensing vs Intuition
- **T/F**: Thinking vs Feeling
- **J/P**: Judging vs Perceiving

## 1. Imports

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
import os

# Dataset download
import kagglehub

# Text processing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Handling class imbalance
from imblearn.over_sampling import RandomOverSampler

# NLTK for stopwords
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("All libraries imported successfully!")

c:\Users\Mr MRF\Desktop\github\MBTI-Text-Classifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


## 2. Dataset Download & Loading

We'll download the MBTI dataset from Kaggle using `kagglehub` and save it to our data folder.

In [ ]:
# Download dataset from Kaggle
print("Downloading MBTI dataset from Kaggle...")
path = kagglehub.dataset_download("datasnaek/mbti-type")
print(f"Dataset downloaded to: {path}")

# Find the CSV file in the downloaded folder
csv_file = os.path.join(path, "mbti_1.csv")

# Load the dataset
df_raw = pd.read_csv(csv_file)

# Save raw data to data folder
os.makedirs('data', exist_ok=True)
df_raw.to_csv('data/raw_data.csv', index=False)
print(f"Raw data saved to data/raw_data.csv")

# Display basic info
print(f"\nDataset shape: {df_raw.shape}")
df_raw.head()

100%|██████████| 24.4M/24.4M [00:05<00:00, 4.70MB/s]

Extracting files...


Dataset downloaded to: C:\Users\Mr MRF\.cache\kagglehub\datasets\datasnaek\mbti-type\versions\1


## 3. Basic Data Exploration

In [ ]:
# Dataset info
print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)
print(f"Number of samples: {len(df_raw)}")
print(f"Number of features: {len(df_raw.columns)}")
print(f"\nColumn names: {df_raw.columns.tolist()}")
print(f"\nData types:\n{df_raw.dtypes}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")

In [ ]:
# Distribution of MBTI types
print("\nMBTI Type Distribution:")
print(df_raw['type'].value_counts())

# Visualize distribution
fig, ax = plt.subplots(figsize=(12, 6))
type_counts = df_raw['type'].value_counts()
colors = plt.cm.viridis(np.linspace(0, 1, len(type_counts)))
bars = ax.bar(type_counts.index, type_counts.values, color=colors)
ax.set_xlabel('MBTI Type', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of MBTI Personality Types', fontsize=14)
plt.xticks(rotation=45)

# Add value labels on bars
for bar, count in zip(bars, type_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nClass imbalance ratio (max/min): {type_counts.max() / type_counts.min():.2f}")

In [ ]:
# Sample post analysis
print("Sample posts from the dataset:")
print("=" * 50)
for i in range(3):
    print(f"\nType: {df_raw.iloc[i]['type']}")
    posts = df_raw.iloc[i]['posts'][:500]  # First 500 chars
    print(f"Posts (first 500 chars): {posts}...")
    print("-" * 50)

## 4. Data Cleaning & Text Aggregation

We'll perform the following text preprocessing steps:
1. Convert to lowercase
2. Remove URLs
3. Remove punctuation and special characters
4. Remove stopwords
5. Aggregate all posts per user into one document (already done in dataset)

In [ ]:
# Load English stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """
    Clean and preprocess text data.
    
    Parameters:
    -----------
    text : str
        Raw text containing posts separated by |||
    
    Returns:
    --------
    str
        Cleaned and aggregated text
    """
    # Remove post separators and combine into one document
    text = text.replace('|||', ' ')
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove MBTI type mentions to prevent data leakage
    mbti_types = ['infp', 'infj', 'intp', 'intj', 'entp', 'enfp', 'istp', 'isfp',
                  'entj', 'istj', 'enfj', 'isfj', 'estp', 'esfp', 'estj', 'esfj']
    for mbti in mbti_types:
        text = re.sub(r'\b' + mbti + r'\b', '', text)
    
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words and len(word) > 2]
    
    return ' '.join(words)

print("Text cleaning function defined.")

In [ ]:
# Apply text cleaning
print("Cleaning text data... This may take a moment.")
df = df_raw.copy()
df['cleaned_posts'] = df['posts'].apply(clean_text)

# Check results
print("\nSample of cleaned text:")
print("=" * 50)
print(f"Original: {df_raw.iloc[0]['posts'][:300]}...")
print(f"\nCleaned: {df.iloc[0]['cleaned_posts'][:300]}...")

# Check for empty posts after cleaning
empty_posts = df['cleaned_posts'].apply(lambda x: len(x) == 0).sum()
print(f"\nEmpty posts after cleaning: {empty_posts}")

# Remove any empty posts
df = df[df['cleaned_posts'].str.len() > 0]
print(f"Dataset size after removing empty posts: {len(df)}")

In [ ]:
# Save processed data
df_processed = df[['type', 'cleaned_posts']].copy()
df_processed.columns = ['type', 'posts']
df_processed.to_csv('data/processed_data.csv', index=False)
print("Processed data saved to data/processed_data.csv")

# Display processed data info
print(f"\nProcessed dataset shape: {df_processed.shape}")
df_processed.head()

## 5. Handling Class Imbalance

The MBTI dataset is imbalanced, with some personality types having significantly more samples than others. We'll use **Random Oversampling** to balance the classes by duplicating samples from minority classes.

In [ ]:
# Prepare features and target
X = df['cleaned_posts'].values
y = df['type'].values

print("Original class distribution:")
print(pd.Series(y).value_counts())

# Visualize original distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original distribution
original_counts = pd.Series(y).value_counts().sort_index()
axes[0].bar(original_counts.index, original_counts.values, color='steelblue')
axes[0].set_xlabel('MBTI Type')
axes[0].set_ylabel('Count')
axes[0].set_title('Original Class Distribution')
axes[0].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Apply Random Oversampling
# We need to reshape X for the oversampler
X_reshaped = X.reshape(-1, 1)

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_reshaped, y)
X_resampled = X_resampled.flatten()

print(f"\nOriginal dataset size: {len(X)}")
print(f"Resampled dataset size: {len(X_resampled)}")

print("\nResampled class distribution:")
print(pd.Series(y_resampled).value_counts())

# Visualize resampled distribution
fig, ax = plt.subplots(figsize=(12, 5))
resampled_counts = pd.Series(y_resampled).value_counts().sort_index()
ax.bar(resampled_counts.index, resampled_counts.values, color='forestgreen')
ax.set_xlabel('MBTI Type')
ax.set_ylabel('Count')
ax.set_title('Class Distribution After Random Oversampling')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. TF-IDF Feature Extraction

We'll convert the text data into numerical features using Term Frequency-Inverse Document Frequency (TF-IDF) vectorization.

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,      # Limit vocabulary size
    min_df=5,               # Ignore terms appearing in fewer than 5 documents
    max_df=0.8,             # Ignore terms appearing in more than 80% of documents
    ngram_range=(1, 2),     # Use unigrams and bigrams
    sublinear_tf=True       # Apply sublinear scaling
)

# Fit and transform training data
print("Fitting TF-IDF vectorizer...")
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data
X_test_tfidf = tfidf.transform(X_test)

print(f"\nTF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (test): {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

# Show some sample features
print(f"\nSample features (first 20): {tfidf.get_feature_names_out()[:20].tolist()}")

## 7. Model Training

We'll train three classic machine learning models:
1. **Logistic Regression** - A simple and interpretable linear model
2. **Support Vector Machine (SVM)** - Effective for high-dimensional text data
3. **Naive Bayes** - Fast and works well with text classification

In [ ]:
# Dictionary to store models and results
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'SVM': LinearSVC(max_iter=2000, random_state=42),
    'Naive Bayes': MultinomialNB()
}

results = {}

# Train each model
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_tfidf, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_tfidf)
    
    # Calculate metrics
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1': f1_score(y_test, y_pred, average='weighted')
    }
    
    print(f"{name} trained successfully!")
    print(f"  Accuracy: {results[name]['accuracy']:.4f}")

print("\nAll models trained!")

## 8. Model Evaluation

We'll evaluate each model using multiple metrics: Accuracy, Precision, Recall, F1-score, and Confusion Matrix.

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results]
})

comparison_df = comparison_df.round(4)
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)

In [ ]:
# Visualize model comparison
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

for i, (name, result) in enumerate(results.items()):
    values = [result['accuracy'], result['precision'], result['recall'], result['f1']]
    ax.bar(x + i * width, values, width, label=name)

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for i, (name, result) in enumerate(results.items()):
    values = [result['accuracy'], result['precision'], result['recall'], result['f1']]
    for j, v in enumerate(values):
        ax.text(j + i * width, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification reports
for name in results:
    print("\n" + "=" * 60)
    print(f"CLASSIFICATION REPORT - {name.upper()}")
    print("=" * 60)
    print(classification_report(y_test, results[name]['predictions']))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

mbti_types = sorted(df['type'].unique())

for ax, (name, result) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, result['predictions'], labels=mbti_types)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=mbti_types, yticklabels=mbti_types,
                annot_kws={'size': 7})
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{name}\nAccuracy: {result["accuracy"]:.4f}')
    ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.show()

## 9. Error Analysis

Let's analyze the types of errors our best model makes to understand its limitations.

In [ ]:
# Find the best model
best_model_name = max(results, key=lambda x: results[x]['f1'])
best_result = results[best_model_name]

print(f"Best Model: {best_model_name}")
print(f"F1-Score: {best_result['f1']:.4f}")

# Get predictions
y_pred_best = best_result['predictions']

# Find misclassified samples
misclassified_idx = np.where(y_test != y_pred_best)[0]
print(f"\nTotal test samples: {len(y_test)}")
print(f"Misclassified samples: {len(misclassified_idx)}")
print(f"Misclassification rate: {len(misclassified_idx)/len(y_test)*100:.2f}%")

In [ ]:
# Analyze most common misclassifications
from collections import Counter

misclassifications = [(y_test[i], y_pred_best[i]) for i in misclassified_idx]
misclass_counts = Counter(misclassifications)

print("\nTop 10 Most Common Misclassifications:")
print("=" * 40)
for (true_type, pred_type), count in misclass_counts.most_common(10):
    print(f"{true_type} -> {pred_type}: {count} times")

In [ ]:
# Analyze per-type accuracy
type_accuracy = {}
for mbti in mbti_types:
    mask = y_test == mbti
    if mask.sum() > 0:
        type_accuracy[mbti] = (y_test[mask] == y_pred_best[mask]).mean()

# Sort by accuracy
type_accuracy_sorted = dict(sorted(type_accuracy.items(), key=lambda x: x[1]))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['crimson' if v < 0.3 else 'orange' if v < 0.5 else 'forestgreen' for v in type_accuracy_sorted.values()]
bars = ax.barh(list(type_accuracy_sorted.keys()), list(type_accuracy_sorted.values()), color=colors)
ax.set_xlabel('Accuracy')
ax.set_ylabel('MBTI Type')
ax.set_title(f'Per-Type Classification Accuracy ({best_model_name})')
ax.set_xlim(0, 1)

# Add value labels
for bar, acc in zip(bars, type_accuracy_sorted.values()):
    ax.text(acc + 0.01, bar.get_y() + bar.get_height()/2, f'{acc:.3f}',
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nBest classified type: {max(type_accuracy, key=type_accuracy.get)} ({max(type_accuracy.values()):.4f})")
print(f"Worst classified type: {min(type_accuracy, key=type_accuracy.get)} ({min(type_accuracy.values()):.4f})")

In [ ]:
# Analyze dimension-level accuracy (E/I, S/N, T/F, J/P)
def get_dimension(mbti_type, dim_idx):
    """Extract a specific dimension from MBTI type"""
    return mbti_type[dim_idx]

dimensions = ['E/I', 'S/N', 'T/F', 'J/P']
dim_accuracy = {}

for i, dim_name in enumerate(dimensions):
    true_dim = np.array([get_dimension(t, i) for t in y_test])
    pred_dim = np.array([get_dimension(t, i) for t in y_pred_best])
    dim_accuracy[dim_name] = (true_dim == pred_dim).mean()

print("\nDimension-Level Accuracy:")
print("=" * 30)
for dim, acc in dim_accuracy.items():
    print(f"{dim}: {acc:.4f}")

# Visualize dimension accuracy
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.Set2(np.linspace(0, 1, 4))
bars = ax.bar(dim_accuracy.keys(), dim_accuracy.values(), color=colors)
ax.set_xlabel('MBTI Dimension')
ax.set_ylabel('Accuracy')
ax.set_title(f'Dimension-Level Classification Accuracy ({best_model_name})')
ax.set_ylim(0, 1)

for bar, acc in zip(bars, dim_accuracy.values()):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.3f}',
            ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

## 10. Conclusion

### Summary of Results

In [ ]:
# Final summary
print("=" * 60)
print("MBTI PERSONALITY CLASSIFICATION - FINAL SUMMARY")
print("=" * 60)

print("\n📊 DATASET:")
print(f"   - Original samples: {len(df_raw)}")
print(f"   - After preprocessing: {len(df)}")
print(f"   - After oversampling: {len(X_resampled)}")
print(f"   - Training set: {len(X_train)}")
print(f"   - Test set: {len(X_test)}")

print("\n🔧 PREPROCESSING:")
print("   - Text cleaning: lowercase, URL removal, punctuation removal")
print("   - Stopword removal using NLTK")
print("   - MBTI type mentions removed to prevent data leakage")
print(f"   - TF-IDF features: {X_train_tfidf.shape[1]}")

print("\n⚖️ CLASS IMBALANCE:")
print("   - Method: Random Oversampling")
print("   - All classes balanced to equal size")

print("\n🤖 MODEL PERFORMANCE:")
for name in comparison_df['Model']:
    row = comparison_df[comparison_df['Model'] == name].iloc[0]
    print(f"   {name}:")
    print(f"      Accuracy: {row['Accuracy']:.4f} | F1-Score: {row['F1-Score']:.4f}")

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   - Accuracy: {best_result['accuracy']:.4f}")
print(f"   - Precision: {best_result['precision']:.4f}")
print(f"   - Recall: {best_result['recall']:.4f}")
print(f"   - F1-Score: {best_result['f1']:.4f}")

print("\n📈 KEY INSIGHTS:")
print(f"   - Best predicted dimension: {max(dim_accuracy, key=dim_accuracy.get)} ({dim_accuracy[max(dim_accuracy, key=dim_accuracy.get)]:.2%})")
print(f"   - Hardest dimension: {min(dim_accuracy, key=dim_accuracy.get)} ({dim_accuracy[min(dim_accuracy, key=dim_accuracy.get)]:.2%})")
print(f"   - Best classified type: {max(type_accuracy, key=type_accuracy.get)}")
print(f"   - Worst classified type: {min(type_accuracy, key=type_accuracy.get)}")

print("\n" + "=" * 60)

### Key Findings

1. **Model Performance**: All three classic ML models (Logistic Regression, SVM, and Naive Bayes) were trained and evaluated on the MBTI dataset.

2. **Class Imbalance**: The original dataset showed significant class imbalance, which was addressed using Random Oversampling.

3. **Feature Extraction**: TF-IDF vectorization with unigrams and bigrams was used to convert text into numerical features.

4. **Challenges**: MBTI classification from text is inherently difficult because:
   - Personality traits are nuanced and may not always be clearly expressed in text
   - The four MBTI dimensions have varying degrees of detectability from writing style
   - Some dimensions (like E/I or T/F) may be easier to detect than others

### Limitations

- Self-reported MBTI types may not be accurate
- Online forum posts may not represent natural writing style
- The dataset is limited to a specific online community

### Future Improvements

- Try different text preprocessing techniques
- Use word embeddings instead of TF-IDF
- Focus on binary classification for individual dimensions
- Collect more diverse data sources

In [ ]:
# Save the best model (optional)
import joblib

os.makedirs('models', exist_ok=True)

# Save model and vectorizer
joblib.dump(best_result['model'], f'models/{best_model_name.lower().replace(" ", "_")}_model.pkl')
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')

print(f"Model saved to: models/{best_model_name.lower().replace(' ', '_')}_model.pkl")
print(f"Vectorizer saved to: models/tfidf_vectorizer.pkl")
print("\n✅ Project completed successfully!")